As you will already made the prediction for each of the scans, here is an example on how to use the bbox to crop out the heart area.

In [6]:
import os, feht
import numpy as np

In [7]:
def add_privileged_annotation(arr: np.ndarray) -> np.ndarray:
    '''
    To leverage the negative cases (the frames without heart presence)
    If the 0 is in the middle of the array, replace as linear interpolation as the nonzero number of its left and right.
    If the 0 is at the beginning, should be replaced by the first nonzero value in the list.
    If the 0 is in the bottom, it should be repalced by the last nonzero value in the list
    '''
    arr = np.array(arr, dtype=float)
    n = len(arr)
    
    non_zero_indices = np.nonzero(arr)[0]
    
    if len(non_zero_indices) == 0:
        return arr
    
    first_non_zero = non_zero_indices[0]
    if first_non_zero > 0:
        arr[:first_non_zero] = arr[first_non_zero]
    
    last_non_zero = non_zero_indices[-1]
    if last_non_zero < n - 1:
        arr[last_non_zero + 1:] = arr[last_non_zero]
    
    zero_indices = np.where(arr == 0)[0]
    
    if len(zero_indices) > 0:
        x = np.arange(n)
        y = arr[non_zero_indices]
        arr[zero_indices] = np.interp(zero_indices, non_zero_indices, y)
        
        return arr

In [8]:
def get_bboxes(
        pred_bbox_path: str, 
        image_shape: tuple[int],
        default_radius: int = 500,
        radius_padding: float = 0.0
        ) -> np.ndarray:
    '''
    The bbox is stored in normalized format, so we need to convert it to the absolute number
    
    :param pred_bbox_path: the path of the prediction bbox
    :param image_height: the height of the image
    :param image_width: the width of the image
    :param default_radius: the default radius of the bbox
    :param radius_padding: the padding of the radius, in normalized format, for ad-hoc adjustment
    '''
    det_prediction = feht.load_pkl(pred_bbox_path)
    image_width, image_height = image_shape

    center_y = det_prediction['center'][:, 0]
    center_x = det_prediction['center'][:, 1]

    radius_arr = det_prediction['radius']
    if np.all(radius_arr == 0):
        radius = default_radius * np.ones_like(det_prediction['center'])   # if nothing from detection, then crop center 500, 500
    else:
        radius_arr += radius_padding
        # radius = 2 * radius_arr * image_shape[0] * image_shape[1] / (image_shape[0] + image_shape[1])
        
        prx = (radius_arr[:, 0] * image_height).astype(int)
        pry = (radius_arr[:, 1] * image_width).astype(int)
        prx = pry = ((prx + pry) / 2).astype(int)
        radius = np.concatenate([prx[:, None], pry[:, None]], axis=1)


    logits_arr = det_prediction['logits']
    class_arr = np.argmax(logits_arr, axis=1)

    center_x *= class_arr  # remove non-heart area
    center_y *= class_arr

    if np.all(np.stack([center_x, center_y]) == 0):
        center_x = 0.5 * np.ones_like(center_x)
        center_y = 0.5 * np.ones_like(center_y)
    else:
        center_x = add_privileged_annotation(center_x)
        center_y = add_privileged_annotation(center_y)

        centers = np.stack([center_x, center_y], axis=1) * np.array([image_width, image_height])
        centers = centers.astype(int)
        radius = radius.astype(int)
        
        x1_y1 = centers - radius
        x2_y2 = centers + radius
        
        bboxes = np.concatenate([x1_y1, x2_y2], axis=1)
        
    return bboxes

The findal results looks like below, it's an N by 4 matrix where N is the frame number and 4 is the top left and botton right points' coordinates, which you can use them for cropping original video frames.

In [ ]:
results = get_bboxes(
    pred_bbox_path="../../temp_output2025-01-31-22-21-15/01-20024+200246.pkl",
    image_shape=(1069, 568)
    )

array([[325,  82, 707, 464],
       [325,  82, 707, 464],
       [325,  82, 707, 464],
       ...,
       [345,  79, 727, 461],
       [345,  79, 727, 461],
       [345,  79, 727, 461]])